## Run in seacells, no extra deps needed

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.legend_handler import HandlerTuple
import matplotlib.lines as mlines
from matplotlib.patches import Patch
import os
from pathlib import Path
import pandas as pd
import seaborn as sns
from statsmodels.stats.multitest import multipletests
from scipy.stats import ttest_ind, pearsonr, spearmanr
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.spatial.distance import pdist

### Getting collapsed datasets for heatmaps

In [ ]:
voom_csv   = "results/downstream_analysis_R/01_DE_beta_pos_immune_neg/voom_logCPM.csv"  # genes × samples
counts_csv = "results/intermediate/pseudobulk_merged_R/pseudobulk_counts_all.csv"                                # to rebuild metadata
out_dir    = "results/downstream_analysis_Python"

# --------------- helpers ---------------
def load_voom(voom_csv):
    E = pd.read_csv(voom_csv, index_col=0)
    E = E.apply(pd.to_numeric, errors="coerce")
    return E

def rebuild_pm_from_counts(counts_csv, E_cols):
    meta_cols = ["sample_id","Condition","laure_region_id","filter"]
    df = pd.read_csv(counts_csv)
    assert all(c in df.columns for c in meta_cols), "counts CSV missing meta cols"
    key = df["sample_id"].astype(str) + "|" + df["laure_region_id"].astype(str) + "|" + df["filter"].astype(str)
    pm = df[meta_cols].copy()
    pm.index = key
    # derive GRADE from laure_region_id like R code
    pm["GRADE"] = pm["laure_region_id"].astype(str).str.extract(r".*_Laure_([^_]+)_.*", expand=False)
    pm = pm.loc[E_cols].copy()  # align to voom columns
    pm["Condition"] = pm["Condition"].astype("category")
    pm["GRADE"]     = pd.Categorical(pm["GRADE"], categories=["0","1","2","3"], ordered=True)
    return pm

def zscore_rows_df(X):
    mu = X.mean(axis=1)
    sd = X.std(axis=1, ddof=0).replace(0, 1.0)
    return X.sub(mu, axis=0).div(sd, axis=0)

In [ ]:
E  = load_voom(voom_csv)
pm = rebuild_pm_from_counts(counts_csv, E.columns)

X  = E.copy()
X  = X[(X > 0).sum(axis=1) > 0]
Xz = zscore_rows_df(X)

cond = pm["Condition"].astype("category")
grad = pm["GRADE"].astype("category")

Xz_T = Xz.T.copy()
Xz_T["Condition"] = cond.values
Xz_T["GRADE"]     = grad.values

agg_condition = Xz_T.groupby("Condition").mean(numeric_only=True).T
agg_grade     = Xz_T.groupby("GRADE").mean(numeric_only=True).T
agg_cxg       = Xz_T.groupby(["Condition","GRADE"]).mean(numeric_only=True).T

agg_condition.to_csv(os.path.join(out_dir, "Heatmap_byCondition.csv"))
agg_grade.to_csv(    os.path.join(out_dir, "Heatmap_byGrade.csv"))
agg_cxg.to_csv(      os.path.join(out_dir, "Heatmap_byCondition_Grade.csv"))


In [ ]:
X_all = E.loc[(E > 0).sum(axis=1) > 0, pm.index]

cond = pm["Condition"].astype("category")
grad = pm["GRADE"].astype("category")

# keep deterministic orders
cond_levels = list(cond.cat.categories)
grad_levels = [g for g in grad.cat.categories if pd.notna(g)]

# ---- Condition × GRADE (mean logCPM per group) ----
Xall_cxg = (
    X_all.T.assign(Condition=cond.values, GRADE=grad.values)
         .groupby(["Condition","GRADE"], observed=True)
         .mean(numeric_only=True)
         .T
)

cols_cxg = []
for c in cond_levels:
    for g in grad_levels:
        if (c, g) in Xall_cxg.columns:
            cols_cxg.append((c, g))
Xall_cxg = Xall_cxg.loc[:, cols_cxg]

Xall_cxg.to_csv(os.path.join(out_dir, "Collapsed_logCPM_allGenes_byCondition_Grade.csv"))

Xall_c = (
    X_all.T.assign(Condition=cond.values)
         .groupby(["Condition"], observed=True)
         .mean(numeric_only=True)
         .T
)

Xall_c = Xall_c.loc[:, cond_levels]
Xall_c.to_csv(os.path.join(out_dir, "Collapsed_logCPM_allGenes_byCondition.csv"))


## Heatmaps generation

In [ ]:
# heatmap_cxg = pd.read_csv('results/downstream_analysis_Python/Collapsed_logCPM_allGenes_byCondition_Grade.csv',index_col=0,header=None).T
# heatmap_cxg = heatmap_cxg.iloc[[9,10,11,12,13,14,15,16,5,6,7,8,3,4,0,1,2]].copy()
# heatmap_cxg.reset_index(drop=True,inplace=True)

# heatmap_c = pd.read_csv('results/downstream_analysis_Python/Collapsed_logCPM_allGenes_byCondition_GradeBinary.csv',index_col=0,header=None).T
# heatmap_c = heatmap_c.iloc[[6,8,4,2,0,7,9,5,3,1]].copy()
# heatmap_c.reset_index(drop=True,inplace=True)

heatmap_c = pd.read_csv('results/downstream_analysis_Python/Collapsed_logCPM_allGenes_byCondition.csv',index_col=0,header=None).T
heatmap_c = heatmap_c.iloc[[3,4,2,0,1]].copy()
heatmap_c.reset_index(drop=True,inplace=True)

# Header
df_heatmap = heatmap_c[['Ins1','Ins2']].T
for col in df_heatmap.columns:
    df_heatmap[col] = df_heatmap[col].astype(float)

df_heatmap__ = df_heatmap.copy()#[[5,6,7,8,9]].copy()
df_heatmap__.columns = ['12w NOD','17w untreated','17w mono aCD3','17w mono GLP1-E2','17w combo aCD3+GLP1-E2']
sns.clustermap(df_heatmap__,#metric='correlation',
               # z_score=0,center=0,
               vmin=0,center=5,vmax=15,cmap='viridis',
               row_cluster = False, col_cluster = False,figsize=(10,6),)
plt.xticks(rotate=60)
plt.show()
# plt.savefig('Heatmap4_fast_JC2.svg')

# Main
df_heatmap = heatmap_c[[
    'Greb1','Xpo1','E2f1','Erbb4','Pgr','Foxa1','S1pr3','Egf',
    'Hspa5','Ern1','Eif2ak3','Atf6','Xbp1','Atf4','Nupr1','Ddit3','Hsp90b1','Calr','Canx','Edem1','Syvn1',
    'Neurod1','Pax6','Nkx6-1','Slc30a8','Iapp','Cpe','Slc2a2','Pcsk1','Mafa','Pdx1','Gck','Trpm5','Glp1r','Foxo1','Ucn3','Mafb','Cd81',
    'Acot7','Itih5','Cmip','Lmo4','Igfbp4','Smad3','Oat','Fxyd1','Mgll','Zyx','Cat','Slc16a1','Smoc2','Igf1','Pdgfra',
    'H2-Q4','H2-K1','H2-Q6','B2m',
    'H2-Aa','H2-Ab1','H2-Eb1','Cd74',
    'Cd274','Stat1','Ccl8','Cxcl10','Cxcl9','Ccl5'
]].T

for col in df_heatmap.columns:
    df_heatmap[col] = df_heatmap[col].astype(float)

sns.clustermap(df_heatmap,metric='correlation',
               z_score=0,
               vmin=-2,center=0,vmax=2,cmap='RdBu_r',
               row_cluster = False, col_cluster = False,figsize=(8,24),)
plt.show()
# plt.savefig('AirportHeatmap3.svg')

## Box-plot graphs for ssGSEA

In [ ]:
ssgsea_df = pd.read_csv('results/downstream_analysis_R/06_ANOVA_beta_pos_immune_neg/tables/ssgsea_scores_manual_plus_reactome_wide.csv')
grade_str = ssgsea_df['GRADE'].astype(str)
ssgsea_df['Grade_binary'] = np.where(grade_str == "0", 0, 1)

cond_order = ['untreated_12w','untreated_17w',
              'mono_aCD3_17w','mono_E2GLP1_17w','combo_aCD3_E2GLP1_17w']
hue_order  = [0, 1]



In [ ]:
# Parameters
pathway = 'REACTOME_CELL_CYCLE'
ylabel = 'Cell Cycle'
filename = 'ssGSEA_CellCycling.svg'

# Custom styling
# → One color per CONDITION
condition_colors = {
    'untreated_12w': '#F2C5A7',
    'untreated_17w': '#EDA89F',
    'mono_aCD3_17w': '#E2696A',
    'mono_E2GLP1_17w': '#8AA9D6',
    'combo_aCD3_E2GLP1_17w': '#9270A5',
}

# → One linestyle per HUE (Grade_binary)
linestyles = ['-', '--']  # e.g., 0 = solid, 1 = dashed

cond_order = [
    'untreated_12w',
    'untreated_17w',
    'mono_aCD3_17w',
    'mono_E2GLP1_17w',
    'combo_aCD3_E2GLP1_17w',
]
hue_order = [0, 1]

plt.figure(figsize=(5,3), dpi=300)
ax = plt.gca()

positions = np.arange(len(cond_order))
width = 0.35  # spacing for hue groups
offsets = np.linspace(-width / 2, width / 2, len(hue_order))

for i, condition in enumerate(cond_order):
    color = condition_colors.get(condition, 'gray')  # fallback color if missing
    for j, grade in enumerate(hue_order):
        subset = ssgsea_df[
            (ssgsea_df['Condition'] == condition)
            & (ssgsea_df['Grade_binary'] == grade)
        ]
        y = subset[pathway].values
        if len(y) == 0:
            continue

        # Jittered x positions
        jitter_strength = 0.05
        jitter = np.random.uniform(-jitter_strength, jitter_strength, size=len(y))
        x_center = positions[i] + offsets[j]
        x = np.ones_like(y) * x_center + jitter

        # Scatter (black points)
        ax.scatter(x, y, color='black', s=2**2, alpha=0.7)

        # Compute box statistics
        q1, med, q3 = np.percentile(y, [25, 50, 75])
        iqr = q3 - q1

        # Robust whiskers (never inside the box)
        low_mask = y >= (q1 - 1.5 * iqr)
        up_mask  = y <= (q3 + 1.5 * iqr)

        if np.any(low_mask):
            lower_whisker = np.min(y[low_mask])
        else:
            lower_whisker = np.min(y)  # fallback when very few points

        if np.any(up_mask):
            upper_whisker = np.max(y[up_mask])
        else:
            upper_whisker = np.max(y)  # fallback

        # Clamp so whiskers are not inside the box
        if lower_whisker > q1:
            lower_whisker = q1
        if upper_whisker < q3:
            upper_whisker = q3


        # Draw box edges (no fill)
        box_x = [x_center - 0.1, x_center + 0.1]
        style = linestyles[j]

        # Box edges
        ax.plot(box_x, [q1, q1], color=color, linestyle=style)
        ax.plot(box_x, [q3, q3], color=color, linestyle=style)
        ax.plot([box_x[0], box_x[0]], [q1, q3], color=color, linestyle=style)
        ax.plot([box_x[1], box_x[1]], [q1, q3], color=color, linestyle=style)

        # Median line
        ax.plot(box_x, [med, med], color=color, linestyle=style)

        # Whiskers
        ax.plot([x_center, x_center], [lower_whisker, q1], color=color, linestyle=style)
        ax.plot([x_center, x_center], [q3, upper_whisker], color=color, linestyle=style)

        # Whisker caps
        ax.plot(
            [x_center - 0.05, x_center + 0.05],
            [lower_whisker, lower_whisker],
            color=color,
            linestyle=style,
        )
        ax.plot(
            [x_center - 0.05, x_center + 0.05],
            [upper_whisker, upper_whisker],
            color=color,
            linestyle=style,
        )

# Aesthetics
ax.set_xticks(positions)
ax.set_xticklabels(cond_order, rotation=30)
ax.set_xlabel('')
ax.set_ylabel(ylabel)

# Legend for hue (linestyle only)
hue_labels = ['No immune infiltration','Immune infiltration present']
legend_handles = [
    mlines.Line2D([], [], color='black', linestyle=linestyles[i], label=hue_labels[i])
    for i in range(len(hue_order))
]
# ax.legend(handles=legend_handles, frameon=True)

plt.tight_layout()
plt.show()


In [ ]:
# Parameters
pathway = 'REACTOME_CELL_CYCLE'
ylabel = 'Cytok'
filename = 'ssGSEA_CellCycle_allgrades.svg'

# Custom styling
condition_colors = {
    'untreated_12w': '#F2C5A7',
    'untreated_17w': '#EDA89F',
    'mono_aCD3_17w': '#E2696A',
    'mono_E2GLP1_17w': '#8AA9D6',
    'combo_aCD3_E2GLP1_17w': '#9270A5',
}

cond_order = [
    'untreated_12w',
    'untreated_17w',
    'mono_aCD3_17w',
    'mono_E2GLP1_17w',
    'combo_aCD3_E2GLP1_17w',
]

plt.figure(figsize=(4,2.85), dpi=300)
ax = plt.gca()

positions = np.arange(len(cond_order))

for i, condition in enumerate(cond_order):
    color = condition_colors.get(condition, 'gray')
    
    # Filter: only grade == 0
    subset = ssgsea_df[
        (ssgsea_df['Condition'] == condition)
        # & (ssgsea_df['Grade_binary'] == 0)
    ]
    y = subset[pathway].values
    if len(y) == 0:
        continue

    # Jittered x positions
    jitter_strength = 0.05
    jitter = np.random.uniform(-jitter_strength, jitter_strength, size=len(y))
    x_center = positions[i]
    x = np.ones_like(y) * x_center + jitter

    # Scatter (black points)
    ax.scatter(x, y, color='black', s=2**2, alpha=0.7)

    # Compute box statistics
    q1, med, q3 = np.percentile(y, [25, 50, 75])
    iqr = q3 - q1

    # Robust whiskers
    low_mask = y >= (q1 - 1.5 * iqr)
    up_mask  = y <= (q3 + 1.5 * iqr)
    lower_whisker = np.min(y[low_mask]) if np.any(low_mask) else np.min(y)
    upper_whisker = np.max(y[up_mask]) if np.any(up_mask) else np.max(y)

    if lower_whisker > q1:
        lower_whisker = q1
    if upper_whisker < q3:
        upper_whisker = q3

    # Draw box edges (no fill)
    box_x = [x_center - 0.1, x_center + 0.1]

    # Box edges
    ax.plot(box_x, [q1, q1], color=color)
    ax.plot(box_x, [q3, q3], color=color)
    ax.plot([box_x[0], box_x[0]], [q1, q3], color=color)
    ax.plot([box_x[1], box_x[1]], [q1, q3], color=color)

    # Median line
    ax.plot(box_x, [med, med], color=color)

    # Whiskers
    ax.plot([x_center, x_center], [lower_whisker, q1], color=color)
    ax.plot([x_center, x_center], [q3, upper_whisker], color=color)

    # Whisker caps
    ax.plot(
        [x_center - 0.05, x_center + 0.05],
        [lower_whisker, lower_whisker],
        color=color,
    )
    ax.plot(
        [x_center - 0.05, x_center + 0.05],
        [upper_whisker, upper_whisker],
        color=color,
    )

# Aesthetics
ax.set_xticks(positions)
ax.set_xticklabels(cond_order, rotation=30)
ax.set_xlabel('')
ax.set_ylabel(ylabel)
ax.set_ylim(0,.195)

plt.tight_layout()
# plt.savefig(filename)
plt.show()


In [ ]:
# Parameters
pathways = ['REACTOME_PROGRAMMED_CELL_DEATH','REACTOME_APOPTOSIS','REACTOME_CYTOKINE_SIGNALING_IN_IMMUNE_SYSTEM','REACTOME_REGULATION_OF_T_CELL_ACTIVATION_BY_CD28_FAMILY']
ylabels = ['ssGSEA Enrichment Score\nfor Programmed Cell Death','ssGSEA Enrichment Score\nfor Apoptosis','ssGSEA Enrichment Score\nfor Cytokine Signaling\nin Immune System','ssGSEA Enrichment Score\nfor Regulation of T cell\nactivation by CD28 family']
filenames= ['ssGSEA_ProgrammedCellDeath.svg','ssGSEA_Apoptosis.svg','ssGSEA_CytokineSignaling.svg','ssGSEA_RegTcell.svg']

splits_colors = [{'untreated_12w': '#F2C5A7', 'untreated_17w': '#EDA89F', 'mono_aCD3_17w': '#E2696A', 'mono_E2GLP1_17w': '#8AA9D6', 'combo_aCD3_E2GLP1_17w': '#9270A5'},
                 {0: '#F2DEC4', 1: '#EDA89F', 2: '#ED7374', 3: '#9270A5'}]

splits_order = [['untreated_12w','untreated_17w','mono_aCD3_17w','mono_E2GLP1_17w','combo_aCD3_E2GLP1_17w'],
                [0,1,2,3]]

for condition_colors,cond_order,prefix in zip(splits_colors,splits_order,['condition_','grade_']):
    for pathway,ylabel,filename in zip(pathways,ylabels,filenames):

        # One color per CONDITION


        plt.figure(figsize=(5, 3), dpi=300)
        ax = plt.gca()

        positions = np.arange(len(cond_order))
        box_halfwidth = 0.1  # controls box width
        jitter_strength = 0.05

        for i, condition in enumerate(cond_order):
            color = condition_colors.get(condition, 'gray')  # fallback color if missing

            # Combine all grades: no hue split
            if prefix=='condition_':
                subset = ssgsea_df[ssgsea_df['Condition'] == condition]
            else:
                subset = ssgsea_df[ssgsea_df['GRADE'] == condition]
                
            y = subset[pathway].dropna().values
            if len(y) == 0:
                continue

            # Jittered x positions
            x_center = positions[i]
            jitter = np.random.uniform(-jitter_strength, jitter_strength, size=len(y))
            x = np.ones_like(y) * x_center + jitter

            # Scatter (black points)
            ax.scatter(x, y, color='black', s=2**2, alpha=0.7)

            # Box stats
            q1, med, q3 = np.percentile(y, [25, 50, 75])
            iqr = q3 - q1
            lower_whisker = np.min(y[y >= q1 - 1.5 * iqr])
            upper_whisker = np.max(y[y <= q3 + 1.5 * iqr])

            # Box (no fill)
            box_x = [x_center - box_halfwidth, x_center + box_halfwidth]

            # Box edges
            ax.plot(box_x, [q1, q1], color=color)
            ax.plot(box_x, [q3, q3], color=color)
            ax.plot([box_x[0], box_x[0]], [q1, q3], color=color)
            ax.plot([box_x[1], box_x[1]], [q1, q3], color=color)

            # Median line
            ax.plot(box_x, [med, med], color=color)

            # Whiskers
            ax.plot([x_center, x_center], [lower_whisker, q1], color=color)
            ax.plot([x_center, x_center], [q3, upper_whisker], color=color)

            # Whisker caps
            ax.plot([x_center - 0.05, x_center + 0.05], [lower_whisker, lower_whisker], color=color)
            ax.plot([x_center - 0.05, x_center + 0.05], [upper_whisker, upper_whisker], color=color)

        # Aesthetics
        ax.set_xticks(positions)
        ax.set_xticklabels(cond_order, rotation=30)
        ax.set_xlabel('')
        ax.set_ylabel(ylabel)

        # No hue legend anymore
        # Optional: add a legend mapping colors to conditions
        # handles = [plt.Line2D([0], [0], color=condition_colors[c], label=c) for c in cond_order]
        # ax.legend(handles=handles, frameon=True, title='Condition')

        plt.tight_layout()
        # plt.savefig(prefix+filename)
        # plt.close()
        plt.show()


In [ ]:
## Figure for Rev3 - marker expression across conditions
df = pd.read_csv('results/downstream_analysis_R/01_DE_beta_pos_immune_neg/voom_logCPM.csv',index_col=0).T
df_meta = pd.read_csv('results/intermediate/pseudobulk_merged_R/pseudobulk_metadata_all.csv')
df_meta['index_'] = df_meta['sample_id']+'|'+df_meta['laure_region_id']+'|'+df_meta['filter']
df['index_'] = df.index
df2 = df.merge(df_meta,on='index_',how='inner')
df2.head()

df_meta = pd.read_csv('results/intermediate/pseudobulk_merged_R/pseudobulk_metadata_all.csv')[['sample_id','laure_region_id','filter','Condition']]
df_meta['index_'] = df_meta['sample_id']+'|'+df_meta['laure_region_id']+'|'+df_meta['filter']
df['index_'] = df.index
df2 = df.merge(df_meta,on='index_',how='inner')
df2_pivot = []
for col in ['Gcg','Irx2','Mafb','Mafa','Ins2','Nkx6-1','Pdx1','Ucn3','Sst','Ppy','Grhl1','Col1a1','Pecam1','Atoh7','Cdh5','Cspg4','Mcam','Krt19','Sox9']:
    df2_ = df2[[col,'Condition','sample_id']].copy()
    df2_.columns = ['expr','cond','sample_id']
    df2_['gene'] = col
    df2_.reset_index(drop=True,inplace=True)
    df2_pivot.append(df2_)
    
df2_pivot = pd.concat(df2_pivot,axis=0)
df2_pivot.head()
# df2_pivot.to_csv('results/rebuttal/df2_pivot2.csv')
# sns.seaborn()

plt.figure(figsize=(15,4))
sns.boxplot(x='gene',y='expr',hue='cond',data=df2_pivot,fliersize=0,hue_order=['untreated_12w','untreated_17w','mono_aCD3_17w','mono_E2GLP1_17w','combo_aCD3_E2GLP1_17w'])
plt.ylabel('Expression (CPM)')
plt.xlabel('')